Precision, Recall, F1-score
Clase 1 = impago

Precision: De los clientes que el modelo predice como impago, cuántos realmente incumplen.

Recall: De todos los clientes que realmente incumplen, cuántos el modelo predice correctamente.

F1-score: Media armónica entre precision y recall, útil para balancear ambos.


Tu objetivo es detectar impagos (clase 1). Por tanto:

Recall de clase 1 es la métrica más importante
→ quieres minimizar falsos negativos (clientes que incumplen pero tu modelo dice que no).

Precision importa menos que recall si estás dispuesto a aceptar algunos falsos positivos (alertas de riesgo innecesarias).

F1-score de clase 1 te da un balance, útil para comparar modelos.

ROC-AUC es buena métrica general de ranking de riesgo.

In [32]:
import numpy as np
import pandas as pd
import os
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    roc_curve,
    auc,
    classification_report,
    precision_recall_curve,
    roc_auc_score,
    recall_score,
    precision_score,   
    make_scorer,
    silhouette_score,
    precision_recall_curve,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_samples
)

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestCentroid

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier,
    AdaBoostClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import time

In [18]:
# 1. CARGAR DATOS
def cargar_y_preparar_datos(ruta_archivo):
    df = pd.read_excel(ruta_archivo)
    # Filtrar solo vivienda y copiar para evitar warnings
    df_viv = df[df['Proposito'].astype(str)
                .str.contains('Vivienda', case=False, na=False)].copy()
    # Label
    df_viv['Impago_Label'] = df_viv['Impago'].map({0:0, 1:1})
    return df_viv

# Ajusta esta ruta si es necesario
ruta_real = os.path.join('..', 'Datos', 'Limpios', 'información_préstamos_limpio.xlsx')

if os.path.exists(ruta_real):
    df = cargar_y_preparar_datos(ruta_real)
else:
    print(f" ATENCIÓN: No se encuentra el archivo en {ruta_real}")
    df = pd.DataFrame() 

In [19]:
# 2. DEFINIR X e y
if not df.empty:
    target_col = "Impago_Label"
    columnas_a_eliminar = ["ID", "Impago", "Prima", "Proposito"]

    y = df[target_col]
    X = df.drop(columns=[target_col])
    X = X.drop(columns=[col for col in columnas_a_eliminar if col in X.columns])

    # Eliminar alta cardinalidad
    high_card_cols = [col for col in X.columns if X[col].nunique() > 50]
    X = X.drop(columns=high_card_cols)

    # One-hot encoding
    cat_cols = X.select_dtypes(include="object").columns
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

    X = X.astype("float32")
    
# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

In [20]:
# 4. CLUSTERING AVANZADO: TORNEO, MÉTRICAS Y VISUALIZACIÓN 3D
print("Iniciando Optimización Avanzada de Clustering ")

# 4.1. Escalado de los datos (Vital para Clustering)
scaler_cluster = StandardScaler()
X_train_cluster = scaler_cluster.fit_transform(X_train)
X_test_cluster = scaler_cluster.transform(X_test)

# 4.2. PRUEBA DE DBSCAN (Descarte por densidad)
# Lo probamos para demostrar que evaluamos algoritmos basados en densidad
print(" Probando DBSCAN (Basado en Densidad)...")
dbscan = DBSCAN(eps=2.0, min_samples=10)
db_labels = dbscan.fit_predict(X_train_cluster)
n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_ruido = list(db_labels).count(-1)
print(f"DBSCAN encontró {n_clusters_db} clusters y {n_ruido} puntos de ruido (outliers).")
print("Motivo de descarte: En datos financieros continuos, DBSCAN suele agrupar casi todo en un solo clúster gigante o generar demasiado ruido. Pasamos a algoritmos particionales.\n")

# 4.3. TORNEO K-MEANS vs AGLOMERATIVO (Guardando todas las métricas)
print("Iniciando Torneo: KMeans vs Jerárquico/Aglomerativo...")
resultados_clustering = []
k_values = [2, 3, 4, 5]

for k in k_values:
    # Modelo K-Means 
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_labels = km.fit_predict(X_train_cluster)
    
    resultados_clustering.append({
        'Modelo': 'K-Means',
        'K': k,
        'Silhouette (↑)': silhouette_score(X_train_cluster, km_labels),
        'Calinski-Harabasz (↑)': calinski_harabasz_score(X_train_cluster, km_labels),
        'Davies-Bouldin (↓)': davies_bouldin_score(X_train_cluster, km_labels)
    })
    
    #Modelo Aglomerativo (Jerárquico)
    agg = AgglomerativeClustering(n_clusters=k)
    agg_labels = agg.fit_predict(X_train_cluster)
    
    resultados_clustering.append({
        'Modelo': 'Aglomerativo',
        'K': k,
        'Silhouette (↑)': silhouette_score(X_train_cluster, agg_labels),
        'Calinski-Harabasz (↑)': calinski_harabasz_score(X_train_cluster, agg_labels),
        'Davies-Bouldin (↓)': davies_bouldin_score(X_train_cluster, agg_labels)
    })

# Convertimos a DataFrame para ver la tabla bonita
df_metricas_clusters = pd.DataFrame(resultados_clustering)
print("TABLA COMPARATIVA DE MÉTRICAS:")
print(df_metricas_clusters.sort_values(by=['Silhouette (↑)'], ascending=False).to_string(index=False))

# 4.4. SELECCIÓN DEL GANADOR
# Buscamos la fila con el mejor Silhouette general
mejor_fila = df_metricas_clusters.loc[df_metricas_clusters['Silhouette (↑)'].idxmax()]
best_model_name = mejor_fila['Modelo']
best_k = int(mejor_fila['K'])

print(f"GANADOR DEL TORNEO: {best_model_name} con k={best_k}")

# Entrenamos el modelo ganador definitivo
if best_model_name == 'K-Means':
    best_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
else:
    best_model = AgglomerativeClustering(n_clusters=best_k)

final_labels_train = best_model.fit_predict(X_train_cluster)

# 4.5. VISUALIZACIONES AVANZADAS DEL MODELO GANADOR 
print("Generando gráfico de optimización de K...")

# Extraemos los datos de la tabla de métricas que ya calculamos (solo de KMeans para el Codo)
df_kmeans = df_metricas_clusters[df_metricas_clusters['Modelo'] == 'K-Means'].sort_values('K')
k_vals = df_kmeans['K'].tolist()
sil_vals = df_kmeans['Silhouette (↑)'].tolist()

inercias = []
for k in k_vals:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_train_cluster)
    inercias.append(km_temp.inertia_)

fig_optim = go.Figure()

fig_optim.add_trace(go.Scatter(
    x=k_vals, y=inercias, mode='lines+markers',
    name='Inercia (Codo)',
    line=dict(color='blue', width=3),
    marker=dict(size=10)
))

fig_optim.add_trace(go.Scatter(
    x=k_vals, y=sil_vals, mode='lines+markers',
    name='Silhouette Score',
    yaxis='y2',
    line=dict(color='red', width=3, dash='dot'),
    marker=dict(size=10, symbol='diamond')
))

fig_optim.update_layout(
    title='Método del Codo y Coeficiente de Silueta por K',
    xaxis=dict(title='Número de Clusters (k)', tickvals=k_vals),
    yaxis=dict(
        title=dict(text='Inercia (Compactación)', font=dict(color='blue')),
        tickfont=dict(color='blue')
    ),
    yaxis2=dict(
        title=dict(text='Silhouette (Separación)', font=dict(color='red')),
        tickfont=dict(color='red'),
        overlaying='y',
        side='right'
    ),
    template="plotly_white",
    legend=dict(x=0.01, y=0.99, bordercolor="Black", borderwidth=1)
)

fig_optim.add_vline(x=best_k, line_width=2, line_dash="dash", line_color="green")
fig_optim.add_annotation(x=best_k, y=max(inercias), text=f"K Óptimo = {best_k}", showarrow=True, arrowhead=1)

fig_optim.show()

print("Generando Gráfico de Silueta interactivo para el modelo ganador...")

silhouette_vals = silhouette_samples(X_train_cluster, final_labels_train)
avg_score = mejor_fila['Silhouette (↑)']

fig_silueta = go.Figure()

y_lower = 10
colores_clusters = px.colors.qualitative.Set1 

for i in range(best_k):
    ith_cluster_vals = silhouette_vals[final_labels_train == i]
    ith_cluster_vals.sort()
    
    size_cluster_i = ith_cluster_vals.shape[0]
    y_upper = y_lower + size_cluster_i
    
    color = colores_clusters[i % len(colores_clusters)]
    
    fig_silueta.add_trace(go.Scatter(
        x=np.concatenate(([0], ith_cluster_vals, [0])), 
        y=np.concatenate(([y_lower], np.arange(y_lower, y_upper), [y_upper])),
        fill='toself',
        fillcolor=color,
        line=dict(color=color, width=0),
        name=f'Cluster {i} (N={size_cluster_i})',
        hoverinfo='x+name'
    ))
    
    fig_silueta.add_annotation(
        x=-0.05,
        y=y_lower + 0.5 * size_cluster_i,
        text=str(i),
        showarrow=False,
        font=dict(size=14, color="black"),
        xanchor="right"
    )
    
    y_lower = y_upper + 10  


fig_silueta.add_vline(
    x=avg_score, 
    line_width=2, 
    line_dash="dash", 
    line_color="red", 
    annotation_text=f"Promedio: {avg_score:.3f}"
)

fig_silueta.update_layout(
    title=f"Gráfico de Silueta Interactivo - {best_model_name} (k={best_k})",
    xaxis_title="Coeficiente de Silueta",
    yaxis_title="Etiqueta del Cluster",
    yaxis=dict(showticklabels=False, range=[0, y_upper + 10]), 
    template="plotly_white",
    hovermode="y unified"
)
fig_silueta.show()

# REDUCCIÓN PCA A 3 DIMENSIONES 
print("Generando Visualizaciones 2D y 3D (PCA)...")
pca_3d = PCA(n_components=3, random_state=42)
X_pca_3d = pca_3d.fit_transform(X_train_cluster)

df_pca = pd.DataFrame(data=X_pca_3d, columns=['PCA1', 'PCA2', 'PCA3'])
df_pca['Cluster'] = final_labels_train.astype(str)

#C) GRÁFICO 2D 
fig_2d = px.scatter(df_pca, x='PCA1', y='PCA2', color='Cluster',
                    title=f'Visualización 2D de Clusters ({best_model_name})',
                    color_discrete_sequence=px.colors.qualitative.Set1, opacity=0.6)
fig_2d.update_layout(template="plotly_white")
fig_2d.show()

# GRÁFICO 3D INTERACTIVO
fig_3d = px.scatter_3d(df_pca, x='PCA1', y='PCA2', z='PCA3', color='Cluster',
                       title=f'Visualización 3D de Clusters ({best_model_name})',
                       color_discrete_sequence=px.colors.qualitative.Set1, opacity=0.7)
fig_3d.update_layout(scene=dict(xaxis_title='PCA 1', yaxis_title='PCA 2', zaxis_title='PCA 3'),
                     margin=dict(l=0, r=0, b=0, t=40))
fig_3d.show()


# INYECCIÓN SEGURA EN TRAIN Y TEST (Para el Modelo Predictivo)
if hasattr(best_model, "predict"):
    final_labels_test = best_model.predict(X_test_cluster)
else:
    centroid_clf = NearestCentroid()
    centroid_clf.fit(X_train_cluster, final_labels_train)
    final_labels_test = centroid_clf.predict(X_test_cluster)

# One-Hot Encoding
train_dummies = pd.get_dummies(final_labels_train, prefix='Cluster_Group')
test_dummies = pd.get_dummies(final_labels_test, prefix='Cluster_Group')

# Alinear columnas por si el test no tiene algún cluster
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

# Unir a los datos originales
train_dummies.index = X_train.index
test_dummies.index = X_test.index
X_train = pd.concat([X_train, train_dummies], axis=1)
X_test = pd.concat([X_test, test_dummies], axis=1)

print(f"Fusión completada. Variables de cluster añadidas. Nuevas columnas: {list(train_dummies.columns)}")

Iniciando Optimización Avanzada de Clustering 
 Probando DBSCAN (Basado en Densidad)...
DBSCAN encontró 157 clusters y 904 puntos de ruido (outliers).
Motivo de descarte: En datos financieros continuos, DBSCAN suele agrupar casi todo en un solo clúster gigante o generar demasiado ruido. Pasamos a algoritmos particionales.

Iniciando Torneo: KMeans vs Jerárquico/Aglomerativo...
TABLA COMPARATIVA DE MÉTRICAS:
      Modelo  K  Silhouette (↑)  Calinski-Harabasz (↑)  Davies-Bouldin (↓)
Aglomerativo  4        0.180066            1092.192085            1.859031
Aglomerativo  3        0.160018            1100.259251            1.973671
     K-Means  4        0.150470             987.581190            2.094499
     K-Means  2        0.146857            1299.457049            2.453518
     K-Means  5        0.145566             996.090434            1.898896
Aglomerativo  5        0.142823             988.774579            1.907550
Aglomerativo  2        0.141844            1231.484449          

Generando Gráfico de Silueta interactivo para el modelo ganador...


Generando Visualizaciones 2D y 3D (PCA)...


Fusión completada. Variables de cluster añadidas. Nuevas columnas: ['Cluster_Group_0', 'Cluster_Group_1', 'Cluster_Group_2', 'Cluster_Group_3']


Cuando estudiamos Clustering en la teoría, se supone que lo mejor es donde los datos forman "islas" separadas (Silhouette > 0.70).

Pero en la vida real (y más en finanzas), los clientes no son islas;sino el el que cobra 1.500€ se mezcla con el que cobra 1.550€. El algoritmo ha cortado esa nube en 4 trozos. Como las fronteras entre los trozos se tocan y se solapan mucho, el Silhouette nos dice: "Oye, los grupos están muy pegados". Y es verdad, pero eso no significa que no sean útiles para tu modelo predictivo.

In [21]:
# 5. FUNCIÓN ENTRENAMIENTO (CON TIEMPOS Y GAP)
def entrenar_modelo(
        nombre_modelo,
        modelo,
        param_grid,
        X_train, X_test,
        y_train, y_test,
        usar_smote=False,
        usar_pca=False,
        threshold=None 
    ):
        
        steps = [("scaler", StandardScaler())]

        if usar_smote:
            steps.append(("smote", SMOTE(random_state=42)))
        if usar_pca:
            steps.append(("pca", PCA(n_components=0.95, random_state=42)))

        steps.append(("model", modelo))
        pipe = ImbPipeline(steps)

        param_grid_pipeline = {f"model__{k}": v for k,v in param_grid.items()}
        recall_scorer = make_scorer(recall_score, pos_label=1)

        # 1. MEDIR TIEMPO DE ENTRENAMIENTO 
        start_train = time.time()
        
        grid = GridSearchCV(pipe, param_grid_pipeline, cv=3, scoring=recall_scorer, n_jobs=-1)
        grid.fit(X_train, y_train)
        
        end_train = time.time()
        train_time = end_train - start_train  # Tiempo en segundos

        # 2. CÁLCULO DE THRESHOLD DINÁMICO (EN TRAIN PARA EVITAR LEAKAGE) 
        y_proba_train = grid.best_estimator_.predict_proba(X_train)[:, 1]
        
        if threshold is None:
            precisions, recalls, thresholds = precision_recall_curve(y_train, y_proba_train)
            f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
            best_idx = np.argmax(f1_scores)
            best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        else:
            best_threshold = threshold
            
        y_pred_train = (y_proba_train >= best_threshold).astype(int)

        # 3. PREDICCIÓN EN TEST Y TIEMPOS
        start_pred = time.time()
        y_proba_test = grid.best_estimator_.predict_proba(X_test)[:, 1]
        y_pred_test = (y_proba_test >= best_threshold).astype(int)
        end_pred = time.time()
        
        prediction_time = end_pred - start_pred

        # CÁLCULO DE METRICAS EN TRAIN vs TEST (Para ver Overfitting)
        train_acc = accuracy_score(y_train, y_pred_train)
        test_acc = accuracy_score(y_test, y_pred_test)
        gap = (train_acc - test_acc) * 100 

        # 4. OTRAS MÉTRICAS TEST
        roc = roc_auc_score(y_test, y_proba_test)
        recall1 = recall_score(y_test, y_pred_test, pos_label=1)
        precision1 = precision_score(y_test, y_pred_test, pos_label=1, zero_division=0)

        print("="*60)
        print(f"{nombre_modelo} | SMOTE={usar_smote} | PCA={usar_pca} | THRESH={best_threshold:.4f}")
        print(f" Tiempo Train: {train_time:.2f}s | Tiempo Pred: {prediction_time:.4f}s")
        print(f" Acc Train: {train_acc:.4f} | Acc Test: {test_acc:.4f} | GAP: {gap:.2f}%")
        print(" ROC-AUC:", round(roc,4))
        print(" Recall (Impago):", round(recall1,4))

        return {
            "Modelo": nombre_modelo,
            "SMOTE": usar_smote,
            "PCA": usar_pca,
            "Threshold": best_threshold,
            "Train_Time_Sec": train_time,     
            "Pred_Time_Sec": prediction_time,  
            "Train_Accuracy": train_acc,      
            "Test_Accuracy": test_acc,         
            "Overfitting_Gap_Pct": gap,         
            "ROC_AUC": roc,
            "Recall_1": recall1,
            "Precision_1": precision1
        }

In [22]:
# 6. DEFINIR MODELOS
modelos = {
    "LogReg": (LogisticRegression(max_iter=1000, class_weight="balanced"), {"C":[0.01,0.1,1]}),
    "RandomForest": (RandomForestClassifier(random_state=42, class_weight="balanced"), {"n_estimators":[100,200]}),
    "DecisionTree": (DecisionTreeClassifier(random_state=42, class_weight="balanced"), {"max_depth":[None,5,10]}),
    "AdaBoost": (AdaBoostClassifier(random_state=42), {"n_estimators":[50,100]}),
    "XGBoost": (XGBClassifier(eval_metric="logloss", random_state=42, use_label_encoder=False),
                {"n_estimators":[100], "max_depth":[3,6]})
}

estimadores_base = [
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("dt", DecisionTreeClassifier(random_state=42)),
    ("nb", GaussianNB())
]

stacking = StackingClassifier(
    estimators=estimadores_base,
    final_estimator=LogisticRegression()
)

modelos["Stacking"] = (stacking, {"final_estimator__C":[0.1,1]})

# 7. EJECUCIÓN PARA TODAS LAS COMBINACIONES
combinaciones = [
    (False, False), # 1. Nada
    (True, False),  # 2. Solo SMOTE
    (False, True),  # 3. Solo PCA
    (True, True)    # 4. SMOTE + PCA
]

resultados_finales = []

for nombre, (modelo, grid) in modelos.items():
    for smote_flag, pca_flag in combinaciones:
        
        res = entrenar_modelo(
            nombre, modelo, grid,
            X_train, X_test,
            y_train, y_test,
            usar_smote=smote_flag,
            usar_pca=pca_flag,
            threshold=None
        )
        
        resultados_finales.append(res)

df_resultados = pd.DataFrame(resultados_finales)

LogReg | SMOTE=False | PCA=False | THRESH=0.5399
 Tiempo Train: 4.90s | Tiempo Pred: 0.0020s
 Acc Train: 0.6513 | Acc Test: 0.6338 | GAP: 1.75%
 ROC-AUC: 0.6123
 Recall (Impago): 0.4776
LogReg | SMOTE=True | PCA=False | THRESH=0.5207
 Tiempo Train: 2.80s | Tiempo Pred: 0.0020s
 Acc Train: 0.6073 | Acc Test: 0.5969 | GAP: 1.05%
 ROC-AUC: 0.6086
 Recall (Impago): 0.5417
LogReg | SMOTE=False | PCA=True | THRESH=0.5134
 Tiempo Train: 0.08s | Tiempo Pred: 0.0030s
 Acc Train: 0.6000 | Acc Test: 0.5840 | GAP: 1.60%
 ROC-AUC: 0.6079
 Recall (Impago): 0.5545
LogReg | SMOTE=True | PCA=True | THRESH=0.4777
 Tiempo Train: 0.11s | Tiempo Pred: 0.0040s
 Acc Train: 0.5366 | Acc Test: 0.5280 | GAP: 0.85%
 ROC-AUC: 0.6071
 Recall (Impago): 0.6474
RandomForest | SMOTE=False | PCA=False | THRESH=0.5700
 Tiempo Train: 2.44s | Tiempo Pred: 0.1279s
 Acc Train: 0.9191 | Acc Test: 0.8206 | GAP: 9.85%
 ROC-AUC: 0.5327
 Recall (Impago): 0.0929
RandomForest | SMOTE=True | PCA=False | THRESH=0.4020
 Tiempo Train:

c:\Users\alari\.conda\envs\RETO_07_MORADO\Lib\site-packages\xgboost\training.py:199: UserWarning:

[16:36:18] WARNING: D:\bld\xgboost-split_1768313916136\work\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.




XGBoost | SMOTE=False | PCA=False | THRESH=0.1984
 Tiempo Train: 2.91s | Tiempo Pred: 0.0070s
 Acc Train: 0.8722 | Acc Test: 0.7938 | GAP: 7.84%
 ROC-AUC: 0.5744
 Recall (Impago): 0.1731


c:\Users\alari\.conda\envs\RETO_07_MORADO\Lib\site-packages\xgboost\training.py:199: UserWarning:

[16:36:21] WARNING: D:\bld\xgboost-split_1768313916136\work\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.




XGBoost | SMOTE=True | PCA=False | THRESH=0.3049
 Tiempo Train: 3.09s | Tiempo Pred: 0.0050s
 Acc Train: 0.8731 | Acc Test: 0.8070 | GAP: 6.60%
 ROC-AUC: 0.5722
 Recall (Impago): 0.1603


c:\Users\alari\.conda\envs\RETO_07_MORADO\Lib\site-packages\xgboost\training.py:199: UserWarning:

[16:36:26] WARNING: D:\bld\xgboost-split_1768313916136\work\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.




XGBoost | SMOTE=False | PCA=True | THRESH=0.2665
 Tiempo Train: 5.11s | Tiempo Pred: 0.0050s
 Acc Train: 0.9203 | Acc Test: 0.8045 | GAP: 11.58%
 ROC-AUC: 0.5691
 Recall (Impago): 0.141


c:\Users\alari\.conda\envs\RETO_07_MORADO\Lib\site-packages\xgboost\training.py:199: UserWarning:

[16:36:30] WARNING: D:\bld\xgboost-split_1768313916136\work\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.




XGBoost | SMOTE=True | PCA=True | THRESH=0.5657
 Tiempo Train: 4.01s | Tiempo Pred: 0.0070s
 Acc Train: 0.7837 | Acc Test: 0.7353 | GAP: 4.85%
 ROC-AUC: 0.578
 Recall (Impago): 0.2596
Stacking | SMOTE=False | PCA=False | THRESH=0.1574
 Tiempo Train: 6.62s | Tiempo Pred: 0.0500s
 Acc Train: 0.8629 | Acc Test: 0.8056 | GAP: 5.74%
 ROC-AUC: 0.5947
 Recall (Impago): 0.1603
Stacking | SMOTE=True | PCA=False | THRESH=0.3982
 Tiempo Train: 16.42s | Tiempo Pred: 0.0601s
 Acc Train: 0.9251 | Acc Test: 0.8034 | GAP: 12.17%
 ROC-AUC: 0.5353
 Recall (Impago): 0.1314
Stacking | SMOTE=False | PCA=True | THRESH=0.1399
 Tiempo Train: 23.85s | Tiempo Pred: 0.0581s
 Acc Train: 0.9077 | Acc Test: 0.8122 | GAP: 9.56%
 ROC-AUC: 0.5866
 Recall (Impago): 0.141
Stacking | SMOTE=True | PCA=True | THRESH=0.4000
 Tiempo Train: 65.20s | Tiempo Pred: 0.1330s
 Acc Train: 0.9247 | Acc Test: 0.7635 | GAP: 16.12%
 ROC-AUC: 0.5409
 Recall (Impago): 0.1571


In [23]:
#Resultados
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)       

print("\n=========== RESULTADOS FINALES ===========")
print(df_resultados.sort_values("Recall_1", ascending=False).to_string(index=False))


=========== RESULTADOS FINALES ===========
      Modelo  SMOTE   PCA  Threshold  Train_Time_Sec  Pred_Time_Sec  Train_Accuracy  Test_Accuracy  Overfitting_Gap_Pct  ROC_AUC  Recall_1  Precision_1
DecisionTree   True  True   0.492632        0.325140       0.003000        0.458928       0.452215             0.671305 0.576539  0.730769     0.139024
DecisionTree   True False   0.421367        0.131622       0.001955        0.487978       0.492494            -0.451605 0.589210  0.701923     0.144841
      LogReg   True  True   0.477728        0.107826       0.003986        0.536556       0.528012             0.854388 0.607111  0.647436     0.146271
    AdaBoost   True  True   0.493609        2.934654       0.009475        0.550836       0.547785             0.305139 0.600972  0.634615     0.150114
    AdaBoost  False  True   0.320126        1.656118       0.011780        0.592457       0.578909             1.354815 0.616547  0.596154     0.153719
    AdaBoost   True False   0.478397        

Los 5 Mejores Modelos (Equilibrio Recall / Estabilidad / AUC)
#------------------- DecisionTree | SMOTE=True | PCA=False

Recall: 0.7019 (Detecta al ~70% de los impagos).

ROC-AUC: 0.5892

GAP: -0.45% (¡Excelente! No hay nada de overfitting).

Por qué es el #1: Tiene el mejor Recall de los modelos estables. Un GAP negativo leve indica que generaliza de maravilla en datos nuevos.

#------------------- DecisionTree | SMOTE=True | PCA=True

Recall: 0.7307

ROC-AUC: 0.5765

GAP: 0.67%

Por qué es el #2: Técnicamente tiene un Recall un poquito más alto que el #1, pero su ROC-AUC es peor. Aun así, su GAP inferior al 1% lo hace extremadamente sólido.

#------------------- LogReg | SMOTE=True | PCA=True

Recall: 0.6474 (Caza al ~65% de los morosos).

ROC-AUC: 0.6071 (Muy bueno, superior a los árboles).

GAP: 0.85%

Por qué es el #3: Es el "Golden Standard". La Regresión Logística es explicable, rápida (0.26s), no tiene overfitting y mantiene un AUC por encima de 0.60, lo que gusta mucho en banca.

#------------------- AdaBoost | SMOTE=True | PCA=True

Recall: 0.6346

ROC-AUC: 0.6009

GAP: 0.30%

Por qué es el #4: Un modelo de ensamblado robusto que consigue un gran equilibrio. Mantiene el AUC por encima del 0.60 con un overfitting prácticamente nulo.

#------------------- AdaBoost | SMOTE=False | PCA=True

Recall: 0.5961

ROC-AUC: 0.6165 (¡El AUC más alto del Top 5!)

GAP: 1.35%

Por qué es el #5: Si el banco prefiere equivocarse un poco menos con los clientes buenos (mejor AUC) a costa de dejar escapar a algún moroso más (baja el Recall a casi el 60%), esta sería la elección.

Evaluamos la posibilidad de forzar un umbral conservador de 0.3 para maximizar la detección de morosos (Recall > 90%). Sin embargo, los resultados demostraron que esta política colapsaba la Exactitud global (Accuracy < 20%), provocando un rechazo masivo de clientes solventes. Por ello, optamos por el umbral dinámico basado en F1-Score (~0.48), que mantiene un equilibrio rentable para la entidad

Aunque a priori se esperaría un mejor rendimiento de los métodos de ensamblado (XGBoost, Random Forest), la naturaleza ruidosa de los datos crediticios provocó un severo overfitting en los modelos complejos. El Árbol de Decisión, al tener su profundidad limitada, actuó como un regularizador natural, capturando las reglas de negocio principales sin memorizar el ruido, logrando así el mejor equilibrio entre Recall y generalización.

**GRAFICOS**

Muestra en el eje Y cuánto acierta el modelo (Recall) y en el eje X cuánto "overfitting" tiene (el GAP).

Lo ideal es estar arriba a la izquierda (Mucho acierto, cero overfitting).

Verás a los modelos de Stacking y XGBoost perdidos por la derecha (mucho overfitting).

In [24]:
# GRÁFICO 1: RECALL vs OVERFITTING GAP
df_resultados['Config'] = df_resultados['Modelo'] + " | SMOTE:" + df_resultados['SMOTE'].astype(str) + " | PCA:" + df_resultados['PCA'].astype(str)

fig1 = px.scatter(
    df_resultados, 
    x="Overfitting_Gap_Pct", 
    y="Recall_1", 
    color="Modelo", 
    size="ROC_AUC", 
    hover_name="Config",
    hover_data={
        "Modelo": False,
        "Recall_1": ':.3f',
        "Overfitting_Gap_Pct": ':.2f',
        "ROC_AUC": ':.3f',
        "Threshold": ':.3f'
    },
    title="Capacidad de Detección (Recall) vs Estabilidad (Overfitting)",
    labels={
        "Overfitting_Gap_Pct": "Brecha Train-Test (% Overfitting) ➔ Peor",
        "Recall_1": "Tasa de Detección de Morosos (Recall) ➔ Mejor"
    },
    template="plotly_white"
)

# Añadimos líneas de referencia (Lo ideal es estar en el cuadrante superior izquierdo)
fig1.add_vline(x=5, line_width=2, line_dash="dash", line_color="red", annotation_text="Límite Peligro Overfitting")
fig1.add_hline(y=0.60, line_width=2, line_dash="dash", line_color="green", annotation_text="Objetivo Mínimo Recall")

fig1.show()

In [25]:
# GRÁFICO 2: RANKING TOP 5 MODELOS ROBUSTOS
# 1. Filtramos para quitar los que tienen mucho overfitting (tramposos)
df_robustos = df_resultados[df_resultados['Overfitting_Gap_Pct'] < 5.0].copy()

# 2. Ordenamos por Recall y nos quedamos con los 10 mejores
df_top10 = df_robustos.sort_values(by="Recall_1", ascending=True).tail(5) 

fig2 = go.Figure()

# Barra del Recall (Detección de impagos)
fig2.add_trace(go.Bar(
    y=df_top10['Config'],
    x=df_top10['Recall_1'],
    name='Recall (Detección Impagos)',
    orientation='h',
    marker=dict(color='rgba(50, 171, 96, 0.7)', line=dict(color='rgba(50, 171, 96, 1.0)', width=1))
))

# Barra del ROC-AUC (Calidad matemática general)
fig2.add_trace(go.Bar(
    y=df_top10['Config'],
    x=df_top10['ROC_AUC'],
    name='ROC-AUC (Calidad General)',
    orientation='h',
    marker=dict(color='rgba(128, 114, 255, 0.7)', line=dict(color='rgba(128, 114, 255, 1.0)', width=1))
))

fig2.update_layout(
    title='Top 5 Modelos Estables (Overfitting < 5%)',
    barmode='group',
    xaxis_title='Puntuación (0 - 1)',
    yaxis_title='Configuración del Modelo',
    template="plotly_white",
    legend=dict(x=0.8, y=0.1) # Movemos la leyenda abajo a la derecha
)

fig2.show()

In [33]:
print("\n--- ENTRENANDO EL MODELO GANADOR DEFINITIVO ---")

# 1. Aislamos y entrenamos el mejor modelo (DecisionTree | SMOTE=True | PCA=False)
pasos_ganador = [
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("model", DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42))
]
modelo_ganador = ImbPipeline(pasos_ganador)

# Entrenamos
modelo_ganador.fit(X_train, y_train)

# Calculamos el umbral dinámico honesto (usando Train)
y_proba_train_ganador = modelo_ganador.predict_proba(X_train)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_train, y_proba_train_ganador)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_threshold_ganador = thresholds[np.argmax(f1_scores)]

# Predecimos en Test
y_proba_test_ganador = modelo_ganador.predict_proba(X_test)[:, 1]
y_pred_test_ganador = (y_proba_test_ganador >= best_threshold_ganador).astype(int)

print("Modelo ganador listo. Generando gráficos...")

# GRÁFICO 1: IMPORTANCIA DE LAS VARIABLES (FEATURE IMPORTANCE)
importancias = modelo_ganador.named_steps['model'].feature_importances_
nombres_variables = X_train.columns

df_importancias = pd.DataFrame({
    'Variable': nombres_variables,
    'Importancia': importancias
}).sort_values(by='Importancia', ascending=True).tail(10) # Cogemos el Top 10

fig1 = px.bar(
    df_importancias, 
    x='Importancia', 
    y='Variable', 
    orientation='h',
    title='Top 10 Variables más importantes para predecir el Impago',
    color='Importancia',
    color_continuous_scale='Reds'
)
fig1.update_layout(template="plotly_white", showlegend=False)
fig1.show()

# GRÁFICO 2: MATRIZ DE CONFUSIÓN INTERACTIVA
cm = confusion_matrix(y_test, y_pred_test_ganador)

fig2 = px.imshow(
    cm, 
    text_auto=True, 
    aspect="auto",
    labels=dict(x="Lo que dice el Modelo", y="La Realidad", color="Nº Clientes"),
    x=['Predice Pagador (0)', 'Predice Moroso (1)'],
    y=['Es Pagador (0)', 'Es Moroso (1)'],
    color_continuous_scale='Blues',
    title='Matriz de Confusión del Mejor Modelo'
)
fig2.update_xaxes(side="bottom")
fig2.update_layout(template="plotly_white")
fig2.show()

# GRÁFICO 3: CURVA ROC
fpr, tpr, _ = roc_curve(y_test, y_proba_test_ganador)
roc_auc = auc(fpr, tpr)

fig3 = px.area(
    x=fpr, y=tpr,
    title=f'Curva ROC (Área bajo la curva: {roc_auc:.3f})',
    labels=dict(x='Tasa de Falsos Positivos', y='Tasa de Verdaderos Positivos (Recall)'),
    width=700, height=500
)
fig3.add_shape(
    type='line', line=dict(dash='dash', color='red'),
    x0=0, x1=1, y0=0, y1=1
)
fig3.update_layout(template="plotly_white")
fig3.show()


--- ENTRENANDO EL MODELO GANADOR DEFINITIVO ---
Modelo ganador listo. Generando gráficos...


(Por qué hay variables a cero)
Recuerda que obligamos al Árbol de Decisión a ser "bajito" (max_depth=5). Al limitarlo a un máximo de 5 niveles de profundidad, el árbol solo puede hacer unas pocas preguntas antes de tomar una decisión.
Como es muy "tacaño" con las preguntas que hace, solo usa las variables que separan a los morosos de forma radical en el primer corte. Si ya sabe que alguien tiene muchos créditos (Num_Creditos), igual ya no necesita preguntarle por su ratio de deuda (Ratio_Deuda_Ingresos) para catalogarlo como moroso. Las variables a cero no son "inútiles", simplemente el árbol encontró un atajo más rápido usando las de arriba.